In [ ]:
# Preparacion y rutas de la actividad
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
import pandas as pd

# Permite ejecutar desde notebooks/, la actividad o la raiz del repositorio.
activity_dir = next(
    path for path in (
        Path.cwd(), Path.cwd().parent, Path.cwd() / "P103_drivers_pandas"
    )
    if (path / "data" / "drivers.csv").is_file()
)
submission_dir = activity_dir / "submission"
submission_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
# Carga de los conductores
drivers = pd.read_csv(activity_dir / "data" / "drivers.csv")
drivers.head()


In [ ]:
# Carga de los registros semanales
timesheet = pd.read_csv(activity_dir / "data" / "timesheet.csv")
timesheet.head()


In [ ]:
# Promedio de horas y millas por conductor
metrics = ["hours-logged", "miles-logged"]
mean_timesheet = timesheet.groupby("driverId")[metrics].mean()
mean_timesheet.head()


In [ ]:
# Totales de horas y millas por conductor: week no es una medida sumable.
sum_timesheet = timesheet.groupby("driverId", as_index=False)[metrics].sum()
sum_timesheet.head()


In [ ]:
# Incorporar el nombre verificando que cada conductor sea unico.
summary = (
    sum_timesheet.merge(
        drivers[["driverId", "name"]], on="driverId", validate="one_to_one"
    )
    .sort_values("driverId")
    .reset_index(drop=True)
)
summary


In [ ]:
# Exportar la tabla sin una columna adicional de indice.
summary.to_csv(submission_dir / "summary.csv", index=False)


In [ ]:
# Los diez conductores con mas millas acumuladas.
# El identificador resuelve empates de manera reproducible.
top10 = summary.sort_values(
    ["miles-logged", "driverId"], ascending=[False, True]
).head(10)
top10


In [ ]:
# Grafico horizontal del ranking.
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top10["name"], top10["miles-logged"], color="tab:orange", alpha=0.8)
ax.invert_yaxis()
ax.set_title("Top 10 conductores por millas recorridas")
ax.set_xlabel("Millas acumuladas")
ax.set_ylabel("Conductor")
ax.xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(submission_dir / "top10_drivers.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
